# 01 · Quickstart: tune a model in one call

`mloptune` wraps a complete, leak-free fine-tuning workflow for any scikit-learn
classifier or regressor (plus XGBoost) behind one configuration object and one call.

Under the hood it follows the **holdout validation** recipe from *Hands-On Machine
Learning*, chapter 1 ("Testing and Validating") and chapter 2 ("Fine-Tune Your Model"):

1. Split the data into **train / validation / test**, stratified for classification.
2. Run an [Optuna](https://optuna.org) study: every trial fits a candidate on the training
   split and is scored on the validation split.
3. Refit the best hyperparameters on **train + validation** ("feeding it more data will
   likely improve its performance", ch. 2).
4. Score that final model **once** on the untouched test split. That number is your
   generalization estimate.

The test split is never seen during tuning, so the workflow avoids the *data snooping*
bias the book warns about.

## Setup

Install the library (from the repository root) with `pip install -e .[test]`.
The two lines below only keep the notebook output tidy.

In [1]:
import warnings

warnings.filterwarnings("ignore")  # hide convergence and progress-bar warnings in tutorial output

import optuna

optuna.logging.set_verbosity(optuna.logging.ERROR)  # Optuna otherwise logs every trial, and a traceback per failed one

In [2]:
import sklearn

import mloptune

print("scikit-learn", sklearn.__version__, "| optuna", optuna.__version__)
print("public API:", mloptune.__all__)

scikit-learn 1.9.1 | optuna 5.0.0
public API: ['FineTuneConfig', 'FineTuneResult', 'FineTuner', 'split_dataset']


## 1. Describe the experiment with `FineTuneConfig`

| Field | Meaning |
|---|---|
| `model_name` | Any scikit-learn classifier/regressor class name, or `XGBClassifier` / `XGBRegressor` |
| `problem_type` | `"classification"` or `"regression"`; the model must match it |
| `experiment_name` | Free text. It is hashed into the random seed, so the same name always gives the same split and the same study |
| `search_space` | Hyperparameters Optuna should tune (see tutorial 02) |
| `n_trials` | Number of Optuna trials |
| `model_kwargs` | Fixed constructor arguments passed to every candidate |
| `scoring` | scikit-learn scorer name; defaults to `accuracy` / `neg_mean_squared_error` |
| `test_size`, `validation_size` | Fractions of the **whole** dataset (defaults 0.2 and 0.2) |

In [3]:
from sklearn.datasets import load_iris

from mloptune import FineTuneConfig, FineTuner

iris = load_iris()

config = FineTuneConfig(
    model_name="RandomForestClassifier",
    problem_type="classification",
    experiment_name="tutorial-01-iris",
    search_space={
        "n_estimators": {"type": "int", "low": 20, "high": 120, "step": 20},
        "max_depth": {"type": "int", "low": 2, "high": 8},
    },
    n_trials=8,
)
config

FineTuneConfig(model_name='RandomForestClassifier', problem_type='classification', experiment_name='tutorial-01-iris', test_size=0.2, validation_size=0.2, scoring=None, n_trials=8, search_space={'n_estimators': {'type': 'int', 'low': 20, 'high': 120, 'step': 20}, 'max_depth': {'type': 'int', 'low': 2, 'high': 8}}, model_kwargs={}, n_startup_trials=None)

## 2. Run it

`FineTuner(config).run(X, y)` does the split, the study, the refit and the final scoring.
`X` can be a NumPy array, a pandas DataFrame, a list of rows or a scipy sparse matrix.

In [4]:
result = FineTuner(config).run(iris.data, iris.target)

print("best hyperparameters :", result.best_params)
print("validation score     :", round(result.validation_score, 4), "(best trial, accuracy)")
print("test score           :", round(result.test_score, 4), "(final model, accuracy)")
print("split sizes          :", result.split_sizes)

best hyperparameters : {'n_estimators': 100, 'max_depth': 3}
validation score     : 1.0 (best trial, accuracy)
test score           : 0.9667 (final model, accuracy)
split sizes          : {'train': 90, 'validation': 30, 'test': 30}


Iris has 150 rows: 20 % (30) go to the test split, 20 % (30) to validation, the remaining
90 to training. The final model was refit on the 120 train + validation rows.

## 3. What you get back

`FineTuneResult` is a frozen dataclass with everything needed to reproduce and reuse the run.

In [5]:
from dataclasses import fields

for f in fields(result):
    value = getattr(result, f.name)
    shown = type(value).__name__ if f.name in ("model", "study") else value
    print(f"{f.name:17s} {shown}")

seed              3066870100
best_params       {'n_estimators': 100, 'max_depth': 3}
validation_score  1.0
test_score        0.9666666666666667
split_sizes       {'train': 90, 'validation': 30, 'test': 30}
model             RandomForestClassifier
study             Study


- `seed` is derived from `experiment_name`. Reuse the name and you get the identical run.
- `model` is a fitted estimator: call `predict`, `predict_proba`, save it with `joblib`.
- `study` is the Optuna study, so every trial is inspectable.

## 4. Use the fitted model

In [6]:
rows = iris.data[[0, 60, 120]]
predicted = result.model.predict(rows)
for row, label in zip(rows, predicted):
    print(row, "->", iris.target_names[label])

[5.1 3.5 1.4 0.2] -> setosa
[5.  2.  3.5 1. ] -> versicolor
[6.9 3.2 5.7 2.3] -> virginica


## 5. Look at every trial

`study.trials_dataframe()` is the equivalent of `GridSearchCV.cv_results_` in the book:
one row per hyperparameter combination that was evaluated.

In [7]:
trials = result.study.trials_dataframe(attrs=("number", "value", "params", "state"))
trials.sort_values("value", ascending=False)

,number,value,params_max_depth,params_n_estimators,state
0,0,1.000000,3,100,COMPLETE
1,1,1.000000,8,40,COMPLETE
2,2,1.000000,5,40,COMPLETE
3,3,1.000000,8,120,COMPLETE
4,4,1.000000,4,60,COMPLETE
6,6,1.000000,5,120,COMPLETE
7,7,1.000000,2,80,COMPLETE
5,5,0.966667,2,20,COMPLETE


## 6. No search space? Then it is a plain train / evaluate run

Leave `search_space` empty to fit `model_kwargs` as-is. You still get the validation
score, the refit and the test score, which makes it a convenient baseline runner.

In [8]:
baseline = FineTuner(
    FineTuneConfig(
        model_name="LogisticRegression",
        problem_type="classification",
        experiment_name="tutorial-01-iris",  # same name -> same split as the forest above
        model_kwargs={"max_iter": 1000},
    )
).run(iris.data, iris.target)

print("logistic regression test accuracy:", round(baseline.test_score, 4))
print("random forest       test accuracy:", round(result.test_score, 4))

logistic regression test accuracy: 1.0
random forest       test accuracy: 0.9667


Because both runs share an `experiment_name`, they were scored on the **same** test rows,
so the two numbers are directly comparable.

## Where next

- **02 · Search spaces** — int / float / categorical specs, Optuna behaviour, failure handling.
- **03 · Classification** — choosing a metric, imbalanced data, evaluating on the test split.
- **04 · Regression** — RMSE, confidence intervals, pandas inputs.
- **05 · Models & XGBoost** — comparing model families, XGBoost, edge cases.
- **06 · Reproducibility & pitfalls** — seeds, validation optimism, saving models.